# 5. Population och stickprov

Hittills har vi *beskrivit* data. Nu tar vi steget till kursens stora fråga:
**vad kan en liten del av en grupp säga om hela gruppen?**

Opinionsinstitut frågar 1 000 personer och uttalar sig om 8 miljoner väljare. Hur kan det fungera?
I den här notebooken bygger du upp svaret genom att göra det man aldrig kan göra i verkligheten:
dra tusentals stickprov ur samma population och se vad som händer.

**Efter den här notebooken kan du**
- skilja på population och stickprov, och på parameter och skattning
- förklara vad urvalsfel är och varför det inte är ett misstag
- beskriva hur stickprovsstorleken påverkar osäkerheten ($1/\sqrt{n}$)
- räkna ut ett medelfel
- förklara vad centrala gränsvärdessatsen säger

**Förkunskaper:** [4. Fördelningar och normalfördelningen](../04_fordelningar/04_fordelningar.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 5.1 Population och parameter

- **Population:** alla individer vi vill uttala oss om.
- **Stickprov:** den delmängd vi faktiskt undersöker.
- **Parameter:** ett mått på populationen. Den är ett fast tal, men nästan alltid **okänd**.
- **Skattning:** samma mått uträknat på stickprovet. Den är känd, men **varierar** mellan stickprov.

| Mått | Population (parameter) | Stickprov (skattning) |
|---|---|---|
| Medelvärde | μ | x̄ |
| Andel | p | p̂ |
| Standardavvikelse | σ | s |

Vi skapar en påhittad population med 10 000 personer. Samma population återkommer i flera av kursens notebooks.

In [ ]:
N = 10_000
population = pd.DataFrame({
    "height": rng.normal(172, 9, N).round(1),                       # cm
    "num_children": rng.poisson(1.5, N),
    "education": rng.choice(["Grundskola", "Gymnasium", "Högskola"], N, p=[0.2, 0.45, 0.35]),
    "income": rng.lognormal(np.log(32), 0.4, N).round(1),           # tkr per månad
})
population.head()

Eftersom vi har skapat populationen själva kan vi tjuvkika på parametrarna. **Det kan man aldrig i verkligheten** –
det är just därför man drar stickprov.

In [ ]:
mu = population["height"].mean()
sigma = population["height"].std(ddof=0)
p = (population["education"] == "Högskola").mean()

print(f"Medellängd i populationen:         μ = {mu:.2f} cm")
print(f"Standardavvikelse i populationen:  σ = {sigma:.2f} cm")
print(f"Andel med högskoleutbildning:      p = {p:.3f}")

## 5.2 Ett stickprov

I ett **obundet slumpmässigt urval (OSU)** har alla individer samma chans att komma med. Vi drar 100 personer:

In [ ]:
sample = population.sample(n=100, random_state=1)      # ← ändra random_state: pröva 2, 3, 4 …

x_bar = sample["height"].mean()
p_hat = (sample["education"] == "Högskola").mean()

print(f"Stickprovets medellängd:  x̄ = {x_bar:.2f} cm   (sant μ = {mu:.2f})   fel: {x_bar - mu:+.2f}")
print(f"Stickprovets andel:       p̂ = {p_hat:.3f}      (sant p = {p:.3f})    fel: {p_hat - p:+.3f}")

🧪 **Labba:** `random_state` är fröet för urvalet. Byt det fem–sex gånger och anteckna x̄ varje gång.
- Får du någonsin exakt μ?
- Ungefär hur långt ifrån brukar du hamna?
- Hamnar du lika ofta över som under?

Skillnaden mellan skattningen och det sanna värdet kallas **urvalsfel**. Det är inget misstag,
utan en ofrånkomlig följd av att vi bara undersöker en del av populationen. Felet är *slumpmässigt*:
ibland för högt, ibland för lågt.

I verkligheten har vi **ett** stickprov och vet inte hur stort felet blev just den gången. Men om vi vet hur mycket
skattningar *brukar* variera, kan vi säga hur stor osäkerheten är. Det undersöker vi nu.

## 5.3 Många stickprov: skattningens fördelning

I stället för att byta frö för hand låter vi datorn dra 1 000 stickprov och spara medelvärdet från vart och ett.

In [ ]:
heights = population["height"].to_numpy()

def sample_means(values, n, num_samples=1000):
    """Dra num_samples stickprov av storlek n och returnera deras medelvärden."""
    return np.array([rng.choice(values, n, replace=False).mean() for _ in range(num_samples)])

means_100 = sample_means(heights, 100)
print("De tio första stickprovsmedelvärdena:", means_100[:10].round(1))

In [ ]:
plt.hist(means_100, bins=30, edgecolor="white")
plt.axvline(mu, color="red", linestyle="--", label=f"sant μ = {mu:.1f}")
plt.xlabel("Stickprovsmedelvärde (cm)")
plt.ylabel("Antal stickprov")
plt.title("Medelvärden från 1 000 stickprov med n = 100")
plt.legend()
plt.show()

print(f"Medelvärdenas medelvärde:      {means_100.mean():.2f}  (μ = {mu:.2f})")
print(f"Medelvärdenas standardavvikelse: {means_100.std():.2f}")

Tre saker att lägga märke till:

1. Medelvärdena **centrerar kring det sanna μ**. Metoden träffar rätt i genomsnitt (den är *väntevärdesriktig*).
2. De flesta stickprov hamnar inom ungefär en centimeter från μ.
3. Histogrammet är **klockformat**.

Det här histogrammet visar **skattningens fördelning** (samplingfördelningen). Blanda inte ihop det med ett histogram
över enskilda personers längd: här är varje "observation" ett helt stickprovs medelvärde.

✏️ **Din tur:** Enskilda personers längd har standardavvikelsen σ ≈ 9 cm. Hur stor andel av de 1 000 stickprovsmedelvärdena
i `means_100` hamnade mer än 2 cm från μ? Hur stor andel av de enskilda personerna i `heights` ligger mer än 2 cm från μ?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print(f"Stickprovsmedelvärden mer än 2 cm från μ: {(np.abs(means_100 - mu) > 2).mean():.1%}")
print(f"Enskilda personer mer än 2 cm från μ:     {(np.abs(heights - mu) > 2).mean():.1%}")
# Ett medelvärde av 100 personer varierar mycket mindre än enskilda personer gör.
```

</details>

## 5.4 Stickprovsstorlekens betydelse

🔮 **Gissa först:** om stickprovet görs fyra gånger så stort (från 100 till 400), blir spridningen bland medelvärdena då
en fjärdedel så stor, hälften så stor, eller något annat?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
for ax, n in zip(axes, [10, 100, 1000]):
    means = sample_means(heights, n)
    ax.hist(means, bins=30)
    ax.axvline(mu, color="red", linestyle="--")
    ax.set_title(f"n = {n}  (spridning {means.std():.2f} cm)")
    ax.set_xlabel("Stickprovsmedelvärde (cm)")
axes[0].set_ylabel("Antal stickprov")
plt.tight_layout()
plt.show()

Större stickprov ger mindre urvalsfel. Men hur mycket mindre? Det får du undersöka själv.

✏️ **Din tur:** Fyll i tabellen genom att köra `sample_means(heights, n).std()` för n = 25, 100, 400 och 1600.
Varje steg är ett fyra gånger så stort stickprov. Vad händer med spridningen i varje steg?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for n in [25, 100, 400, 1600]:
    spread = sample_means(heights, n).std()
    print(f"n = {n:>4}: spridning {spread:.2f} cm")
# Spridningen halveras (ungefär) varje gång n fyrdubblas.
```

</details>

### Medelfelet

Mönstret du just hittade har en formel. Standardavvikelsen för ett stickprovsmedelvärde kallas **medelfel**
(på engelska *standard error*):

$$\text{medelfel} = \frac{\sigma}{\sqrt{n}}$$

Det är alltså **roten ur n** som styr. Fyra gånger så stort stickprov ger halva osäkerheten.
För att få en tiondel så stor osäkerhet behövs *hundra gånger* så stort stickprov.

In [ ]:
print(f"{'n':>5} {'simulerad spridning':>20} {'σ/√n':>8}")
for n in [10, 25, 100, 400, 1000]:
    simulated = sample_means(heights, n).std()
    print(f"{n:>5} {simulated:>20.2f} {sigma / np.sqrt(n):>8.2f}")

Formeln stämmer väl med simuleringen. (Vid n = 1000 ligger simuleringen lite under formeln. Det beror på att vi då
har dragit en tiondel av hela populationen, och ett stickprov utan återläggning kan inte avvika lika mycket.
När populationen är mycket större än stickprovet, vilket är det vanliga, märks det inte.)

> **Det här är förklaringen till opinionsmätningarna.** Osäkerheten beror på stickprovets storlek, inte på hur stor
> andel av populationen man frågar. 1 000 slumpvis valda personer ger samma precision för Sverige som för hela EU.

I verkligheten känner vi inte σ. Då skattar vi den med stickprovets standardavvikelse *s* och räknar $s/\sqrt{n}$.
Det räcker alltså med **ett enda stickprov** för att uppskatta hur mycket skattningen varierar:

In [ ]:
s = sample["height"].std()
std_error = s / np.sqrt(len(sample))
print(f"Från vårt enda stickprov: x̄ = {x_bar:.2f} cm,  s = {s:.2f} cm,  medelfel = {std_error:.2f} cm")

Jämför med spridningen du såg i histogrammet för n = 100 i avsnitt 5.3. Vi fick ungefär samma tal, utan att behöva dra
1 000 stickprov. Det är grunden för konfidensintervallen i [notebook 7](../07_konfidensintervall/07_konfidensintervall.ipynb).

## 5.5 Centrala gränsvärdessatsen

Längd är normalfördelad, så det är kanske inte överraskande att medelvärdena också blev det.
Men inkomsterna i populationen är kraftigt högersneda. Hur fördelar sig *medelinkomsten* i ett stickprov?

🔮 **Gissa först:** blir histogrammet över stickprovsmedelvärden också snett?

In [ ]:
incomes = population["income"].to_numpy()
n = 5        # ← ändra: pröva 2, 5, 30 och 200

means = sample_means(incomes, n)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(incomes, bins=60)
axes[0].set_title("Populationen: enskilda inkomster")
axes[0].set_xlabel("Inkomst (tkr per månad)")
axes[0].set_ylabel("Antal personer")

axes[1].hist(means, bins=30, density=True, alpha=0.7)
x = np.linspace(means.min(), means.max(), 200)
axes[1].plot(x, stats.norm.pdf(x, means.mean(), means.std()), color="red", label="Normalfördelning")
axes[1].set_title(f"Medelvärden av stickprov, n = {n}")
axes[1].set_xlabel("Medelinkomst (tkr per månad)")
axes[1].set_ylabel("Täthet")
axes[1].legend()
plt.tight_layout()
plt.show()

🧪 **Labba:** Öka `n` steg för steg. Vid vilket n tycker du att histogrammet följer den röda kurvan?

**Centrala gränsvärdessatsen (CGS):** medelvärdet av ett tillräckligt stort slumpmässigt stickprov är ungefär
normalfördelat, **oavsett hur populationen är fördelad**.

En vanlig tumregel är n ≥ 30, men mycket sned data kräver mer. CGS är skälet till att normalfördelningen är så central
i statistiken: metoderna i de kommande notebookarna handlar om medelvärden och andelar, och de blir ungefär normalfördelade
även när datan inte är det.

✏️ **Din tur:** Antal barn (`num_children`) är varken symmetriskt eller kontinuerligt. Rita ett histogram över 1 000
stickprovsmedelvärden med n = 50. Ser det klockformat ut?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
children = population["num_children"].to_numpy()
plt.hist(sample_means(children, 50), bins=30, edgecolor="white")
plt.xlabel("Medelantal barn i stickprovet")
plt.ylabel("Antal stickprov")
plt.show()
```

</details>

## 5.6 Andelar fungerar på samma sätt

En andel är egentligen ett medelvärde av ettor och nollor (minns tricket från notebook 1). Därför gäller allt ovan också för andelar.
Medelfelet för en andel är

$$\text{medelfel} = \sqrt{\frac{p(1-p)}{n}}$$

In [ ]:
has_degree = (population["education"] == "Högskola").to_numpy()
n = 100

p_hats = sample_means(has_degree, n)

plt.hist(p_hats, bins=np.arange(0.145, 0.565, 0.01), edgecolor="white")
plt.axvline(p, color="red", linestyle="--", label=f"sant p = {p:.3f}")
plt.xlabel("Stickprovsandel p̂")
plt.ylabel("Antal stickprov")
plt.legend()
plt.show()

print(f"Simulerad spridning: {p_hats.std():.3f}")
print(f"Formeln:             {np.sqrt(p * (1 - p) / n):.3f}")

Med n = 100 varierar en andel alltså med ungefär ±5 procentenheter (en standardavvikelse) mellan stickprov.
Andelar är "dyra" att skatta noggrant.

✏️ **Din tur:** Ett opinionsinstitut vill att medelfelet för ett parti med p ≈ 0,25 ska vara högst 0,01 (en procentenhet).
Pröva dig fram med formeln: hur många personer behöver de fråga?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for n in [500, 1000, 1500, 1875, 2000]:
    print(f"n = {n:>4}: medelfel {np.sqrt(0.25 * 0.75 / n):.4f}")
# Det behövs ungefär 1 900 personer. (Löser man ut n ur formeln: n = p(1−p) / 0,01² = 1875.)
```

</details>

## Sammanfattning

- En **parameter** (μ, p) beskriver populationen och är okänd. En **skattning** (x̄, p̂) räknas ut på stickprovet.
- Skattningen träffar aldrig exakt: skillnaden är **urvalsfelet**, och det är slumpmässigt.
- **Medelfelet** $\sigma/\sqrt{n}$ anger hur mycket ett stickprovsmedelvärde typiskt varierar. Fyra gånger så stort stickprov ger halva osäkerheten.
- **Centrala gränsvärdessatsen:** stickprovsmedelvärden är ungefär normalfördelade även om datan inte är det.

Tillsammans säger de två sista punkterna: *x̄ är ungefär normalfördelat kring μ med standardavvikelsen σ/√n.*
Den meningen är grunden för allt som kommer härnäst.

### Kontrollfrågor
1. Vad är skillnaden mellan standardavvikelse och medelfel?
2. Ett stickprov på 400 personer har medelfelet 0,5. Hur stort stickprov behövs för medelfelet 0,25?
3. En undersökning av 1 000 personer i Sverige och en av 1 000 personer i Kina – vilken är mest precis?
4. Allt i den här notebooken förutsätter en sak om hur stickprovet har valts ut. Vilken?

<details>
<summary>💡 <b>Svar</b></summary>

1. Standardavvikelsen beskriver hur mycket *enskilda individer* varierar. Medelfelet beskriver hur mycket ett *stickprovsmedelvärde* varierar mellan olika stickprov.
2. 1 600. Halva medelfelet kräver fyra gånger så stort stickprov.
3. Ungefär lika precisa. Precisionen beror på n, inte på populationens storlek.
4. Att urvalet är slumpmässigt. Vad som händer annars är ämnet för nästa notebook.

</details>

**Nästa notebook:** [6. Datakvalitet, felkällor och bias](../06_datakvalitet_och_bias/06_datakvalitet_och_bias.ipynb)